In [1]:
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Any, Optional
import re, json

from dotenv import load_dotenv
load_dotenv(Path("configs") / "local.env")
import torch
torch.manual_seed(3647)
torch.set_float32_matmul_precision('high')
from transformers import set_seed
set_seed(42)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

In [11]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model_name = "Qwen/Qwen3-8B"

print(f"--> device: {device}, model_name: {model_name}")

--> device: cuda, model_name:Qwen/Qwen3-8B


In [3]:
# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)

#quant = BitsAndBytesConfig(
#    load_in_4bit=True,
#    bnb_4bit_compute_dtype=torch.bfloat16,
#    bnb_4bit_use_double_quant=True,
#    bnb_4bit_quant_type="nf4",
#)

quant = BitsAndBytesConfig(load_in_8bit=True, bnb_8bit_compute_dtype=torch.bfloat16)
#quant = None

In [4]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto",
    quantization_config=quant,
)

tools = [
    {
        "name": "get_weather",
        "description": "Get weather by location",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

In [5]:
def chat(messages, max_new_tokens=1024, thinking=True, tools=None, temperature=0.2):
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=thinking, # 若你确认模型是 thinking 版本再开；普通指令版请注释掉
        tools=tools,              # 若这里报参名错误，换成 functions=tools
    )

    inputs_ids = tokenizer([prompt], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **inputs_ids,
        max_new_tokens=max_new_tokens, # max_new_tokens=32768
        temperature=temperature,       # 放到 generate 里，而不是 apply_chat_template
        do_sample=(temperature > 0),   # 温度>0时启用采样
        eos_token_id=tokenizer.eos_token_id,
    )

    output_ids = generated_ids[0][len(inputs_ids.input_ids[0]):].tolist() 
    # index = len(output_ids) - output_ids[::-1].index(151668) # </think>
    # output_ids[:index]

    text = tokenizer.decode(output_ids, skip_special_tokens=True)

    return text.strip()

In [6]:
def _as_tool(obj: Any) -> Optional[Dict[str, Any]]:
    ok = isinstance(obj, dict) and "name" in obj and "arguments" in obj
    if not ok:
        return None

    # arguments 允许是 str（未解析的 JSON），也允许是 dict
    args = obj["arguments"]
    if isinstance(args, str): # 尝试把字符串 arguments 转为 dict
        try:
            args = json.loads(args)
        except Exception:
            pass

    return { "name": obj["name"], "arguments": args }

def parse_tool_call(text: str):
    """
    兼容两种常见输出：
    1) 纯 JSON：{"name":"get_weather","arguments":{"city":"Shanghai"}}
    2) 带标签：<|tool_call|>{"name":...}</|tool_call|> 或 <tool_call>...</tool_call>
    """

    m = re.search(r"<\|?tool_call\|?>\s*(\{.*?\})\s*</\|?tool_call\|?>", text, flags=re.S)
    if m:
        text = m.group(1)

    text = text.strip()
    last_brace = text.rfind("}") # 尝试截断到最后一个 '}'（避免结尾带多余自然语言）
    if last_brace != -1:
        text = text[:last_brace+1]

    try:
        obj = json.loads(text)
        if "name" in obj and "arguments" in obj:
            return obj
    except Exception:
        pass

    return None

def parse_tool_calls(text: str) -> List[Dict[str, Any]]:
    """
    从模型输出中解析出 0..N 个工具调用：
    return: [{"name": str, "arguments": dict|str}, ...]
    """
    results: List[Dict[str, Any]] = []

    # 1) 先抓所有标签形式的调用（支持 <tool_call> 与 <|tool_call|>）
    tag_pattern = r"<\|?tool_call\|?>\s*(\{.*?\})\s*</\|?tool_call\|?>"
    for m in re.finditer(tag_pattern, text.strip(), flags=re.S):
        block = m.group(1).strip()
        # 截到最后一个右花括号，避免尾部自然语言
        block = block[: block.rfind("}") + 1] if "}" in block else block
        try:
            obj = json.loads(block)
            tool = _as_tool(obj)
            if tool:
                results.append(tool)
        except Exception:
            continue

    if results:
        return results

    # 2) 没有标签：尝试纯 JSON（对象或数组），先粗暴截到最后一括号，剔除结尾赘语
    if "}" in s:
        s = s[:s.rfind("}") + 1]

    try:
        obj = json.loads(s)
        if isinstance(obj, dict):
            tool = _as_tool(obj)
            if tool:
                return [tool]
        elif isinstance(obj, list):
            for it in obj: # 数组中每个元素应该是一个 tool 对象
                tool = _as_tool(it)
                if tool:
                    results.append(tool)
            if results:
                return results
    except Exception:
        pass

    # 3) 兜底：有些模型会输出多个 JSON 对象串联，用粗略查找提取每段 {...}, 可能匹配到非工具 JSON，解析后再用 _as_tool 过滤
    brace_pattern = r"\{(?:[^{}]|(?R))*\}"  # 递归样式在部分引擎不可用；退而求其次：
    approx_pattern = r"\{[\s\S]*?\}"        # 简化版：匹配看起来像 JSON 的块（容错，不完美）
    for m in re.finditer(approx_pattern, text):
        chunk = m.group(0)
        try:
            obj = json.loads(chunk)
            tool = _as_tool(obj)
            if tool:
                results.append(tool)
        except Exception:
            continue

    return results

In [7]:
msgs = [
    {"role": "user", "content": "Give me a short introduction to large language model."},
]

print(chat(msgs))

/home/appuser/.local/lib/python3.11/site-packages/bitsandbytes/autograd/_functions.py:186: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


<think>
Okay, the user wants a short introduction to large language models. Let me start by defining what they are. They're AI systems trained on vast amounts of text data, right? So I should mention that they're based on deep learning and natural language processing.

Wait, I need to highlight their key features. They can understand and generate human-like text, handle multiple languages, and perform tasks like answering questions, writing stories, coding. Maybe mention their scale—like having billions of parameters. That's important for their capabilities.

Also, applications are a big part. They're used in chatbots, virtual assistants, content creation, data analysis. Should I include examples like GPT or BERT? Maybe not too specific, but it's good to note they're part of the transformer architecture. Oh, and the training process involves massive datasets and computational resources. That's a key point.

I should keep it concise. Avoid jargon but make sure it's informative. Let me s

In [8]:
msgs = [
    {"role": "user", "content": "简要的介绍一下大语言模型(large language models)。"},
]

print(chat(msgs))

<think>
嗯，用户让我简要介绍一下大语言模型。首先，我需要确定用户的需求是什么。他们可能是在学习AI的基础知识，或者想了解最新的技术趋势，也可能是想应用这些模型到自己的项目中。不过用户要求的是“简要”，所以不能太深入，但也不能太浅显。

接下来，我得回忆一下大语言模型的基本定义。大语言模型是基于深度学习的神经网络，特别是Transformer架构，通过大量文本数据进行训练，能够理解和生成自然语言。需要提到它们的规模，比如参数量巨大，通常在数亿到数千亿之间。

然后，要说明它们的能力，比如文本生成、问答、翻译、编程等。可能还要提到它们的应用场景，比如聊天机器人、内容创作、数据分析等。不过用户可能更关注核心概念，而不是具体应用，所以应用部分可以简略带过。

还要注意区分大语言模型和其他类型的模型，比如传统的NLP模型，强调其规模和预训练的重要性。可能需要提到预训练和微调的区别，但保持简洁。

另外，用户可能对技术细节不太熟悉，所以需要用通俗易懂的语言，避免过多术语。比如，解释Transformer架构时，可以简单说它是一种高效的结构，适合处理长文本。

还要考虑用户可能的深层需求。他们可能想知道大语言模型的优势，比如强大的语言理解和生成能力，或者它们的局限性，比如对数据的依赖和潜在的偏见问题。不过用户要求简要，所以可能不需要深入讨论优缺点，但可以稍微提及。

最后，确保结构清晰，分点说明，比如定义、核心技术、能力、应用、挑战等。但保持段落简短，避免信息过载。检查是否有遗漏的重要点，比如训练数据的多样性和规模，以及模型的持续发展，如GPT、BERT等具体例子是否需要提及。不过用户要的是简要，所以可能不需要具体例子，但可以提到一些知名模型作为补充。

总结一下，回答需要包括定义、核心技术（Transformer、预训练）、能力、应用、挑战，保持简洁明了，适合不同背景的读者理解。
</think>

大语言模型（Large Language Models, LLMs）是基于深度学习的神经网络模型，通过海量文本数据进行训练，能够理解和生成自然语言。其核心特点包括：

1. **规模庞大**：通常包含数十亿至数千亿参数，通过大规模预训练捕捉语言规律和知识。

2. **多任务能力**：可完成文本生成、问答、翻译、编程、逻辑推理等任务，支持多种语言。

3. **预训练与微调

In [10]:
messages = [
    #{ "role": "system", "content": "Only return the tool call JSON with keys: name, arguments; no extra text." },
    { "role": "user", "content": "What's the weather in Shanghai?" },
]

text = chat(messages, thinking=False, tools=tools)
print(f"<-- response: \n{text}\n")
print(f"<-- parse_tool_call:\n{parse_tool_call(text)}\n")
print(f"<-- parse_tool_calls:\n{parse_tool_calls(text)}\n")

<-- response: 
<tool_call>
{"name": "get_weather", "arguments": {"city": "Shanghai"}}
</tool_call>

<-- parse_tool_call:
{'name': 'get_weather', 'arguments': {'city': 'Shanghai'}}

<-- parse_tool_calls:
[{'name': 'get_weather', 'arguments': {'city': 'Shanghai'}}]

